<a href="https://colab.research.google.com/github/sreerajmk/railroad/blob/genAi/StockPredictionUsingGenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import datetime

# Fetch historical stock price data using yfinance
start_date = datetime.datetime(2020, 1, 1)
end_date = datetime.datetime(2023, 1, 1)
stock_data = yf.download('AAPL', start=start_date, end=end_date)

# Preprocess data
stock_data = stock_data[['Close']]  # Use closing prices
stock_data = stock_data.values  # Convert to numpy array

[*********************100%***********************]  1 of 1 completed


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

# VAE model
def sampling(args):
    z_mean, z_log_var = args
    batch = K.shape(z_mean)[0]
    dim = K.int_shape(z_mean)[1]
    epsilon = K.random_normal(shape=(batch, dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

original_dim = stock_data.shape[1]
latent_dim = 2

# Encoder
inputs = Input(shape=(original_dim,))
h = Dense(16, activation='relu')(inputs)
z_mean = Dense(latent_dim)(h)
z_log_var = Dense(latent_dim)(h)
z = Lambda(sampling, output_shape=(latent_dim,))([z_mean, z_log_var])

# Decoder
decoder_h = Dense(16, activation='relu')
decoder_mean = Dense(original_dim, activation='sigmoid')
h_decoded = decoder_h(z)
x_decoded_mean = decoder_mean(h_decoded)

# VAE model
vae = Model(inputs, x_decoded_mean)
vae.compile(optimizer='adam', loss='mse')

# Train VAE
vae.fit(stock_data, stock_data, epochs=50, batch_size=16, shuffle=True)

# Encode data
encoder = Model(inputs, z_mean)
encoded_data = encoder.predict(stock_data)

Epoch 1/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 16834.4336
Epoch 2/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 16960.7656
Epoch 3/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17122.8242
Epoch 4/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17167.7520
Epoch 5/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17261.5938
Epoch 6/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17452.2363
Epoch 7/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17326.8516
Epoch 8/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 16749.5098
Epoch 9/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 16940.4492
Epoch 10/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 16750.2305
Epoch 11/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17135.9473
Epoch 12/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17107.7852
Epoch 13/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17246.2773
Epoch 14/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 17042.2266
Epoch 15/50
48/48 ━━━━━━━━━━━

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.layers import LSTM, Dropout
from tensorflow.keras.models import Sequential

# Scale data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(encoded_data)

# Prepare data for LSTM
def create_dataset(data, time_step=1):
    X, Y = [], []
    for i in range(len(data) - time_step - 1):
        a = data[i:(i + time_step), :]
        X.append(a)
        Y.append(data[i + time_step, :])
    return np.array(X), np.array(Y)

time_step = 10
X, Y = create_dataset(scaled_data, time_step)

# Reshape input to be [samples, time steps, features]
X = X.reshape(X.shape[0], X.shape[1], X.shape[2])

# LSTM model
model = Sequential()
model.add(LSTM(units=50, return_sequences=True, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.2))
model.add(LSTM(units=50, return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(units=Y.shape[1]))

model.compile(optimizer='adam', loss='mean_squared_error')

# Train LSTM
model.fit(X, Y, epochs=100, batch_size=32, verbose=1)

# Predict
predicted_stock_price = model.predict(X)
predicted_stock_price = scaler.inverse_transform(predicted_stock_price)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - loss: 0.1920
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0137
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0082
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0085
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0078
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0060
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0072
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0063
Epoch 9/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0067
Epoch 10/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0063
Epoch 11/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0065
Epoch 12/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0056
Epoch 13/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0057
Epoch 14/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 0.0060
Epoch 15/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - lo

In [ ]:
# Print the predicted stock prices
print("Predicted Stock Prices:")
for i, price in enumerate(predicted_stock_price):
    print(f"Day {i + 1}: {price[0]:.2f}")